In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
PROJECT_DIR = Path.cwd().resolve().parent
DATA_DIR = PROJECT_DIR / 'data'
SRC_DIR = PROJECT_DIR / 'src'
EXTRACT_DIR = PROJECT_DIR / 'processed' / 'extraction'
OUTPUT_DIR = PROJECT_DIR / 'processed' / 'transform'
REPORT_DIR = PROJECT_DIR / 'reports'

print('PROJECT_DIR:', PROJECT_DIR)
print('EXTRACT_DIR:', EXTRACT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

## 데이터 로드

In [ ]:
# all events (chart + lab + eMAR medication point events)
all_events = pd.read_csv(EXTRACT_DIR / 'all_events_long.csv')

print(f"all_events shape: {all_events.shape}")
print(f"Columns: {all_events.columns.tolist()}")
all_events.head()


In [ ]:
# adm_pat_icu (icu stay info)
adm_pat_icu = pd.read_csv(EXTRACT_DIR / 'adm_pat_icu.csv')
print(f"adm_pat_icu shape: {adm_pat_icu.shape}")
print(f"Columns: {adm_pat_icu.columns.tolist()}")
adm_pat_icu.head()


In [ ]:
# Date columns
all_events['charttime'] = pd.to_datetime(all_events['charttime'], errors='coerce')
for col in ['intime', 'outtime', 'admittime', 'dischtime', 'deathtime']:
    adm_pat_icu[col] = pd.to_datetime(adm_pat_icu[col], errors='coerce')

## VALUE 변환 (문자열 → 숫자)

In [ ]:
all_events.rename(columns={'value': 'value_str',
                           'valuenum': 'value'}, inplace=True)

In [ ]:
# 문자열/범주형 -> 숫자 변환

# Delirium assessment: Positive=1, Negative=0, UTA/other=NaN
all_events.loc[all_events['value_str'].astype('string').str.lower().eq('positive'), 'value'] = 1
all_events.loc[all_events['value_str'].astype('string').str.lower().eq('negative'), 'value'] = 0

# RASS text values
all_events.loc[all_events['value_str'] == '"+4 Combative' , 'value']        = 4
all_events.loc[all_events['value_str'] == '+3 Pulls or removes tube(s) or catheter(s); aggressive', 'value'] = 3
all_events.loc[all_events['value_str'] == '"+2 Frequent nonpurposeful movement', 'value']                    = 2
all_events.loc[all_events['value_str'] == '"+1 Anxious', 'value']           = 1
all_events.loc[all_events['value_str'] == ' 0  Alert and calm', 'value']    = 0
all_events.loc[all_events['value_str'] == '-1 Awakens to voice (eye opening/contact) > 10 sec', 'value'] = -1
all_events.loc[all_events['value_str'] == '"-2 Light sedation' , 'value']   = -2
all_events.loc[all_events['value_str'] == '"-3 Moderate sedation', 'value'] = -3
all_events.loc[all_events['value_str'] == '"-4 Deep sedation', 'value']     = -4
all_events.loc[all_events['value_str'] == '"-5 Unarousable' , 'value']      = -5


## 단위 변환

In [ ]:
if 'valueuom_original' not in all_events.columns:
    all_events['valueuom_original'] = all_events['valueuom']

def fahr_to_celsius(temp_fahr):
    return (temp_fahr - 32) * 5 / 9

# Temperature: Fahrenheit -> Celsius
temp_f_mask = all_events['label'].eq('Temperature Fahrenheit')
all_events.loc[temp_f_mask, 'value'] = fahr_to_celsius(all_events.loc[temp_f_mask, 'value'])
all_events.loc[all_events['feature_name'].eq('temperature'), 'valueuom'] = 'C'

# Weight: lbs -> kg
weight_lbs_mask = all_events['label'].eq('Admission Weight (lbs.)')
all_events.loc[weight_lbs_mask, 'value'] = all_events.loc[weight_lbs_mask, 'value'] * 0.453592
all_events.loc[all_events['feature_name'].eq('weight'), 'valueuom'] = 'kg'

# Height: inch -> cm
height_inch_mask = all_events['label'].eq('Height')
all_events.loc[height_inch_mask, 'value'] = all_events.loc[height_inch_mask, 'value'] * 2.54
all_events.loc[all_events['feature_name'].eq('height'), 'valueuom'] = 'cm'


In [ ]:
all_events.to_csv(OUTPUT_DIR / 'all_events_filtered.csv', index=False)


## Delirium assessment 간격 확인

In [ ]:
# 같은 ICU stay 안에서 delirium assessment가 몇 시간 간격으로 기록되는지 확인
delirium_assessment_times = (
    all_events
    .loc[
        all_events['feature_name'].eq('delirium') & all_events['charttime'].notna(),
        ['subject_id', 'hadm_id', 'stay_id', 'charttime']
    ]
    .drop_duplicates(['stay_id', 'charttime'])
    .sort_values(['stay_id', 'charttime'])
    .reset_index(drop=True)
)

delirium_assessment_times['assessment_interval_hours'] = (
    delirium_assessment_times
    .groupby('stay_id')['charttime']
    .diff()
    .dt.total_seconds()
    .div(3600)
)

delirium_interval_hours = delirium_assessment_times['assessment_interval_hours'].dropna()

delirium_interval_summary = pd.Series({
    'n_assessment_timepoints': len(delirium_assessment_times),
    'n_stays_with_assessment': delirium_assessment_times['stay_id'].nunique(),
    'n_intervals': len(delirium_interval_hours),
    'mean_interval_hours': delirium_interval_hours.mean(),
    'median_interval_hours': delirium_interval_hours.median(),
})
stay_level_interval_summary = (
    delirium_assessment_times
    .groupby('stay_id')['assessment_interval_hours']
    .agg(
        n_intervals='count',
        mean_interval_hours='mean',
        median_interval_hours='median',
    )
    .query('n_intervals > 0')
    .reset_index()
)
display(delirium_interval_summary.to_frame('value'))

## 시간 계산 (ICU 입실 기준)

In [ ]:
# length of stay in icu (hours)
adm_pat_icu['los_hours'] = ((adm_pat_icu['outtime'] - adm_pat_icu['intime']).dt.total_seconds() / 3600).astype(int)

In [ ]:
patient_cols = [
    'stay_id', 'subject_id', 'hadm_id', 'anchor_age', 'gender', 'los_hours',
    'admission_type', 'race', 'specialty', 'hospital_expire_flag', 'intime', 'outtime'
]
patient_cols = [c for c in patient_cols if c in adm_pat_icu.columns]
merge_keys = [c for c in ['stay_id', 'subject_id', 'hadm_id'] if c in all_events.columns and c in patient_cols]
all_events = all_events.merge(adm_pat_icu[patient_cols], on=merge_keys, how='inner')

all_events


## 12시간 구간 라벨링 (long-format events)

In [ ]:
# 12시간 단위 라벨링
all_events['_elapsed_hours'] = (
    (all_events['charttime'] - all_events['intime']).dt.total_seconds() / 3600.0
)

all_events = all_events.loc[
    (all_events['_elapsed_hours'] >= 0) &
    (all_events['charttime'] <= all_events['outtime'])
].copy()

all_events['bin'] = np.ceil((all_events['_elapsed_hours'] - 1e-9) / 12).clip(lower=1).astype('int64')
all_events['hours'] = all_events['bin'] * 12
all_events.drop(columns='_elapsed_hours', inplace=True)

# event row는 집계/pivot 없이 시간순으로 유지
sort_cols = ['stay_id', 'charttime', 'bin', 'feature_name', 'itemid']
timeseries = all_events.sort_values(sort_cols).reset_index(drop=True)

preview_cols = ['subject_id', 'hadm_id', 'stay_id', 'charttime', 'bin', 'hours', 'feature_name', 'value', 'valueuom']

print("\n=== Long-format events with 12-hour labels ===")
print(f"Unique stay_ids: {timeseries['stay_id'].nunique()}")
print(f"Total event rows: {len(timeseries):,}")
display(timeseries[preview_cols].head(20))
timeseries.to_csv(OUTPUT_DIR / 'all_events_12h_long.csv', index=False)
print(f"Saved: all_events_12h_long.csv ({len(timeseries):,} rows, {timeseries['stay_id'].nunique()} stays)")

## 12시간 구간 라벨링 (wide-format by charttime)

In [ ]:
wide_index_cols = [c for c in [
    'subject_id', 'hadm_id', 'stay_id', 'charttime', 'bin', 'hours',
    'age', 'gender', 'los_hours', 'admission_type', 'race', 'specialty', 'hospital_expire_flag',
    'intime', 'outtime'
] if c in timeseries.columns]

wide_source = (
    timeseries
    .loc[timeseries['feature_name'].notna(), wide_index_cols + ['feature_name', 'value']]
    .sort_values([c for c in ['stay_id', 'charttime', 'bin', 'feature_name'] if c in timeseries.columns])
)

duplicate_key_cols = wide_index_cols + ['feature_name']
duplicate_feature_rows = wide_source.duplicated(duplicate_key_cols, keep=False).sum()
if duplicate_feature_rows:
    print(f"Same stay/charttime/feature duplicate rows: {duplicate_feature_rows:,}; keeping the first observed value for exact duplicates only.")
    wide_source = wide_source.drop_duplicates(duplicate_key_cols, keep='first')

wide_timeseries = (
    wide_source
    .pivot(index=wide_index_cols, columns='feature_name', values='value')
    .reset_index()
    .sort_values([c for c in ['stay_id', 'charttime', 'bin'] if c in wide_index_cols])
    .reset_index(drop=True)
)
wide_timeseries.columns.name = None

# height/weight는 가장 처음 측정된 값 사용
static_fill_cols = ['height', 'weight']
static_fill_cols = [col for col in static_fill_cols if col in wide_timeseries.columns]

for col in static_fill_cols:
    first_values = (
        wide_timeseries
        .sort_values(['stay_id', 'charttime'])
        .groupby('stay_id')[col]
        .transform(lambda values: values.dropna().iloc[0] if values.notna().any() else np.nan)
    )
    wide_timeseries[col] = first_values

wide_timeseries


## 처치/장치 노출 추가

In [ ]:
# procedure/device interval과 charttime이 실제로 겹치는 row만 1로 표시
procedure_events_path = EXTRACT_DIR / 'procedure_selected.csv'
procedure_events = pd.read_csv(procedure_events_path)

for col in ['starttime', 'endtime', 'intime', 'outtime']:
    if len(procedure_events) and col in procedure_events.columns:
        procedure_events[col] = pd.to_datetime(procedure_events[col], errors='coerce')

In [ ]:
procedure_intervals = pd.DataFrame(columns=['stay_id', 'start_use', 'end_use', 'feature_name'])

proc = procedure_events.copy()
proc['start_use'] = proc['starttime']
proc['end_use'] = proc['endtime'].fillna(proc['starttime'])
proc = proc.loc[
    proc['feature_name'].notna()
    & proc['start_use'].notna()
    & proc['end_use'].notna()
    & proc['intime'].notna()
    & proc['outtime'].notna()
].copy()
proc['start_use'] = proc[['start_use', 'intime']].max(axis=1)
proc['end_use'] = proc[['end_use', 'outtime']].min(axis=1)
proc = proc.loc[proc['start_use'] <= proc['end_use']].copy()
procedure_intervals = (
    proc[['stay_id', 'start_use', 'end_use', 'feature_name']]
    .drop_duplicates()
    .sort_values(['stay_id', 'start_use', 'end_use', 'feature_name'])
    .reset_index(drop=True)
)

procedure_binary_cols = sorted(procedure_intervals['feature_name'].dropna().unique().tolist())
for col in procedure_binary_cols:
    wide_timeseries[col] = 0

for stay_id, stay_proc in procedure_intervals.groupby('stay_id'):
    stay_idx = wide_timeseries.index[wide_timeseries['stay_id'].eq(stay_id)]
    if len(stay_idx) == 0:
        continue
    stay_charttime = wide_timeseries.loc[stay_idx, 'charttime']
    for row in stay_proc.itertuples(index=False):
        exposed_idx = stay_charttime.index[
            (stay_charttime >= row.start_use) & (stay_charttime <= row.end_use)
        ]
        if len(exposed_idx):
            wide_timeseries.loc[exposed_idx, row.feature_name] = 1

for col in procedure_binary_cols:
    wide_timeseries[col] = wide_timeseries[col].astype('int8')

print(f"Procedure/device columns added: {procedure_binary_cols}")
if procedure_binary_cols:
    display(wide_timeseries.loc[wide_timeseries[procedure_binary_cols].sum(axis=1) > 0, ['stay_id', 'charttime', 'bin', 'hours'] + procedure_binary_cols].head(10))


## delirium 라벨 생성


In [ ]:
# 12시간 window 단위 delirium label 생성
# 해당 bin 안 assessment 중 하나라도 positive면 1, assessment는 있지만 positive가 없으면 0, assessment가 없으면 null.
if 'delirium' in wide_timeseries.columns:
    wide_timeseries.drop(columns='delirium', inplace=True)

delirium_12h = (
    timeseries
    .loc[timeseries['feature_name'].eq('delirium') & timeseries['value'].notna(), ['stay_id', 'bin', 'value']]
    .groupby(['stay_id', 'bin'], as_index=False)['value']
    .max()
    .rename(columns={'value': 'delirium'})
)
delirium_12h['delirium'] = delirium_12h['delirium'].astype('int8')

wide_timeseries = wide_timeseries.merge(delirium_12h, on=['stay_id', 'bin'], how='left')
wide_timeseries['delirium'] = wide_timeseries['delirium'].astype('Int8')

print('12-hour delirium label 분포:')
print(delirium_12h['delirium'].value_counts())
display(wide_timeseries.head(20))

In [ ]:
# stay_id 단위로 delirium이 ever positive였는지 여부 계산

subject_ever_delirium = (
    wide_timeseries
    .dropna(subset=['delirium'])
    .groupby('subject_id')['delirium']
    .max()
    .reindex(wide_timeseries['subject_id'].dropna().unique(), fill_value=0)
    .fillna(0)
    .astype('int8')
)

wide_timeseries['ever_delirium'] = wide_timeseries['subject_id'].map(subject_ever_delirium).fillna(0).astype('int8')


print('ever_delirium subject 분포:')
print(subject_ever_delirium.value_counts().sort_index())
display(wide_timeseries.head(50))

wide_timeseries.to_csv(OUTPUT_DIR / 'all_events_12h_wide_by_charttime.csv', index=False)
print(f"Saved: all_events_12h_wide_by_charttime.csv ({len(wide_timeseries):,} rows, {wide_timeseries['stay_id'].nunique()} stays)")

## 12시간 binning

변수별 측정 빈도 확인

In [ ]:
# Median measurement interval by variable for chart/lab events
chart_lab_times = (
    timeseries
    .loc[
        timeseries['source_table'].isin(['chartevents', 'labevents'])
        & timeseries['feature_name'].notna()
        & timeseries['charttime'].notna()
        & timeseries['value'].notna(),
        ['source_table', 'feature_name', 'subject_id', 'charttime']
    ]
    .drop_duplicates(subset=['source_table', 'feature_name', 'subject_id', 'charttime'])
    .sort_values(['source_table', 'feature_name', 'subject_id', 'charttime'])
    .copy()
)

chart_lab_times['interval_hours'] = (
    chart_lab_times
    .groupby(['source_table', 'feature_name', 'subject_id'])['charttime']
    .diff()
    .dt.total_seconds() / 3600
)

chart_lab_interval_medians = (
    chart_lab_times
    .dropna(subset=['interval_hours'])
    .groupby(['source_table', 'feature_name'], as_index=False)['interval_hours']
    .median()
    .rename(columns={'interval_hours': 'median_interval_hours'})
    .sort_values(['source_table', 'feature_name'])
)

display(chart_lab_interval_medians)


빈번한 V/S, glucose 등은 요약정보 사용.

lab 결과는 bin 내에서 가장 최근 값 사용. 값이 없다면 직전 bin에서 측정된 값으로 imputation. (그럼에도 불구하고 결측치 많은 변수는 추후 train set에서 확인 후 제거 예정)

In [ ]:
# 12-hour bin-level wide table using extraction_variable_catalog.csv
# - binning == 'aggregation': mean/median/std/count/min/max/latest within the bin
# - binning == 'most recent': latest value in the bin, then carry-forward from previous bins
# - binning == 'at least once': 1 if present/exposed at least once within the bin
# - binning == 'static': baseline/static values repeated across bins

variable_catalog = pd.read_csv(SRC_DIR / 'extraction_variable_catalog.csv')
variable_catalog['binning'] = variable_catalog['binning'].str.strip().str.lower()

feature_binning = (
    variable_catalog
    .dropna(subset=['feature_name', 'binning'])
    [['feature_name', 'binning']]
    .drop_duplicates()
)

procedure_catalog_features = set(
    variable_catalog.loc[variable_catalog['source_table'].eq('procedureevents'), 'feature_name'].dropna()
)
aggregation_features = sorted(feature_binning.loc[feature_binning['binning'].eq('aggregation'), 'feature_name'].unique())
most_recent_features = sorted(feature_binning.loc[feature_binning['binning'].eq('most recent'), 'feature_name'].unique())
static_features_from_events = sorted(
    set(feature_binning.loc[feature_binning['binning'].eq('static'), 'feature_name'])
    - {'age', 'gender'}
)
at_least_once_features = sorted(
    set(feature_binning.loc[feature_binning['binning'].eq('at least once'), 'feature_name'])
    - procedure_catalog_features
)
procedure_features = sorted(
    variable_catalog.loc[
        variable_catalog['source_table'].eq('procedureevents')
        & variable_catalog['binning'].eq('at least once'),
        'feature_name'
    ].dropna().unique()
)

adm_for_bin = adm_pat_icu.copy()
for col in ['intime', 'outtime']:
    adm_for_bin[col] = pd.to_datetime(adm_for_bin[col], errors='coerce')
adm_for_bin['los_hours'] = (adm_for_bin['outtime'] - adm_for_bin['intime']).dt.total_seconds() / 3600
adm_for_bin['total_bins'] = np.ceil(adm_for_bin['los_hours'].fillna(0).clip(lower=0) / 12).astype('int64')
if 'anchor_age' in adm_for_bin.columns and 'age' not in adm_for_bin.columns:
    adm_for_bin['age'] = adm_for_bin['anchor_age']

base_bin_rows = adm_for_bin.loc[adm_for_bin['total_bins'] > 0].copy()
base_bin_rows = base_bin_rows.loc[base_bin_rows.index.repeat(base_bin_rows['total_bins'])].copy()
base_bin_rows['bin'] = base_bin_rows.groupby('stay_id').cumcount() + 1
base_bin_rows['hours'] = base_bin_rows['bin'] * 12
base_bin_rows['bin_start'] = base_bin_rows['intime'] + pd.to_timedelta((base_bin_rows['bin'] - 1) * 12, unit='h')
base_bin_rows['bin_end'] = base_bin_rows['intime'] + pd.to_timedelta(base_bin_rows['bin'] * 12, unit='h')

base_cols = [c for c in [
    'subject_id', 'hadm_id', 'stay_id', 'bin', 'hours', 'bin_start', 'bin_end',
    'age', 'gender', 'los_hours', 'admission_type', 'race', 'specialty', 'hospital_expire_flag',
    'intime', 'outtime'
] if c in base_bin_rows.columns]
wide_timeseries_12h_bin = base_bin_rows[base_cols].sort_values(['stay_id', 'bin']).reset_index(drop=True)

# Static event-derived features, e.g. height/weight: first observed value per stay.
static_event_values = (
    timeseries
    .loc[
        timeseries['feature_name'].isin(static_features_from_events)
        & timeseries['charttime'].notna()
        & timeseries['value'].notna(),
        ['stay_id', 'feature_name', 'charttime', 'value']
    ]
    .sort_values(['stay_id', 'feature_name', 'charttime'])
    .drop_duplicates(['stay_id', 'feature_name'], keep='first')
)
if len(static_event_values):
    static_event_values = static_event_values.pivot(index='stay_id', columns='feature_name', values='value').reset_index()
    static_event_values.columns.name = None
    wide_timeseries_12h_bin = wide_timeseries_12h_bin.merge(static_event_values, on='stay_id', how='left')

# Aggregated features: summary statistics plus latest value within each bin.
aggregation_source = timeseries.loc[
    timeseries['feature_name'].isin(aggregation_features)
    & timeseries['charttime'].notna()
    & timeseries['value'].notna(),
    ['stay_id', 'bin', 'feature_name', 'charttime', 'value']
].copy()

aggregation_summary = (
    aggregation_source
    .groupby(['stay_id', 'bin', 'feature_name'])['value']
    .agg(['mean', 'median', 'std', 'count', 'min', 'max'])
)
aggregation_latest = (
    aggregation_source
    .sort_values(['stay_id', 'bin', 'feature_name', 'charttime'])
    .drop_duplicates(['stay_id', 'bin', 'feature_name'], keep='last')
    .set_index(['stay_id', 'bin', 'feature_name'])['value']
    .rename('latest')
)
aggregation_summary = aggregation_summary.join(aggregation_latest, how='left')

if len(aggregation_summary):
    aggregation_summary = aggregation_summary.unstack('feature_name')
    aggregation_summary.columns = [f'{feature}_{stat}' for stat, feature in aggregation_summary.columns]
    aggregation_summary = aggregation_summary.reset_index()
    wide_timeseries_12h_bin = wide_timeseries_12h_bin.merge(aggregation_summary, on=['stay_id', 'bin'], how='left')

# Most-recent features: latest result in each bin, forward-filled within stay.
most_recent_source = (
    timeseries
    .loc[
        timeseries['feature_name'].isin(most_recent_features)
        & timeseries['charttime'].notna()
        & timeseries['value'].notna(),
        ['stay_id', 'bin', 'feature_name', 'charttime', 'value']
    ]
    .sort_values(['stay_id', 'bin', 'feature_name', 'charttime'])
    .drop_duplicates(['stay_id', 'bin', 'feature_name'], keep='last')
)
most_recent_wide = most_recent_source.pivot(index=['stay_id', 'bin'], columns='feature_name', values='value').reset_index()
if len(most_recent_wide):
    most_recent_wide.columns.name = None
    most_recent_cols = [c for c in most_recent_wide.columns if c not in ['stay_id', 'bin']]
    wide_timeseries_12h_bin = wide_timeseries_12h_bin.merge(most_recent_wide, on=['stay_id', 'bin'], how='left')
    wide_timeseries_12h_bin[most_recent_cols] = (
        wide_timeseries_12h_bin
        .sort_values(['stay_id', 'bin'])
        .groupby('stay_id')[most_recent_cols]
        .ffill()
    )

# At-least-once point features, e.g. medication and delirium assessment.
at_least_once_source = timeseries.loc[
    timeseries['feature_name'].isin(at_least_once_features)
    & timeseries['value'].notna(),
    ['stay_id', 'bin', 'feature_name', 'value']
].copy()
at_least_once_wide = (
    at_least_once_source
    .groupby(['stay_id', 'bin', 'feature_name'])['value']
    .max()
    .unstack('feature_name')
    .reset_index()
)
if len(at_least_once_wide):
    at_least_once_wide.columns.name = None
    at_least_once_cols = [c for c in at_least_once_wide.columns if c not in ['stay_id', 'bin']]
    wide_timeseries_12h_bin = wide_timeseries_12h_bin.merge(at_least_once_wide, on=['stay_id', 'bin'], how='left')
    wide_timeseries_12h_bin[at_least_once_cols] = wide_timeseries_12h_bin[at_least_once_cols].fillna(0).astype('int8')

# Previous-bin delirium status for sequential modeling. The first 12-hour bin has
# no pre-ICU previous state, so it is coded as 0.
if 'delirium' in wide_timeseries_12h_bin.columns:
    wide_timeseries_12h_bin['prev_delirium'] = (
        wide_timeseries_12h_bin
        .sort_values(['stay_id', 'bin'])
        .groupby('stay_id')['delirium']
        .shift(1)
        .fillna(0)
        .astype('int8')
    )
else:
    wide_timeseries_12h_bin['delirium'] = 0
    wide_timeseries_12h_bin['prev_delirium'] = 0

# Procedure/device intervals: positive if the procedure interval overlaps the 12-hour bin.
procedure_events = pd.read_csv(EXTRACT_DIR / 'procedure_selected.csv')
for col in ['starttime', 'endtime', 'intime', 'outtime']:
    if col in procedure_events.columns:
        procedure_events[col] = pd.to_datetime(procedure_events[col], errors='coerce')

proc = procedure_events.loc[procedure_events['feature_name'].isin(procedure_features)].copy()
proc['start_use'] = proc['starttime']
proc['end_use'] = proc['endtime'].fillna(proc['starttime'])
proc = proc.loc[
    proc['stay_id'].notna()
    & proc['feature_name'].notna()
    & proc['start_use'].notna()
    & proc['end_use'].notna()
    & proc['intime'].notna()
    & proc['outtime'].notna()
].copy()
proc['start_use'] = proc[['start_use', 'intime']].max(axis=1)
proc['end_use'] = proc[['end_use', 'outtime']].min(axis=1)
proc = proc.loc[proc['start_use'] <= proc['end_use']].copy()
proc['start_bin'] = np.ceil(((proc['start_use'] - proc['intime']).dt.total_seconds() / 3600 - 1e-9) / 12).clip(lower=1).astype('int64')
proc['end_bin'] = np.ceil(((proc['end_use'] - proc['intime']).dt.total_seconds() / 3600 - 1e-9) / 12).clip(lower=1).astype('int64')

procedure_bin_rows = []
for row in proc[['stay_id', 'feature_name', 'start_bin', 'end_bin']].drop_duplicates().itertuples(index=False):
    for bin_id in range(row.start_bin, row.end_bin + 1):
        procedure_bin_rows.append((row.stay_id, bin_id, row.feature_name, 1))

procedure_bin = pd.DataFrame(procedure_bin_rows, columns=['stay_id', 'bin', 'feature_name', 'value'])
if len(procedure_bin):
    procedure_bin = (
        procedure_bin
        .groupby(['stay_id', 'bin', 'feature_name'])['value']
        .max()
        .unstack('feature_name')
        .reset_index()
    )
    procedure_bin.columns.name = None
    procedure_cols = [c for c in procedure_bin.columns if c not in ['stay_id', 'bin']]
    wide_timeseries_12h_bin = wide_timeseries_12h_bin.merge(procedure_bin, on=['stay_id', 'bin'], how='left')
    wide_timeseries_12h_bin[procedure_cols] = wide_timeseries_12h_bin[procedure_cols].fillna(0).astype('int8')

wide_timeseries_12h_bin = wide_timeseries_12h_bin.sort_values(['stay_id', 'bin']).reset_index(drop=True)
wide_timeseries_12h_bin.to_csv(OUTPUT_DIR / 'all_events_12h_binned.csv', index=False)

print(f"Saved: all_events_12h_binned.csv ({len(wide_timeseries_12h_bin):,} rows, {wide_timeseries_12h_bin['stay_id'].nunique()} stays)")
print('Binning rules from catalog:')
print(feature_binning['binning'].value_counts().sort_index())
display(wide_timeseries_12h_bin.head(20))



## 포함/제외 기준 적용

In [ ]:
def _cohort_window_counts(cohort_df):
    # LSTM modeling is framed on 12-hour steps: 4 input steps, then delirium
    # prediction for the 4th input step plus the next 2 steps. A 72-hour ICU stay
    # is the minimum 6-step span. Candidate windows exclude the first 48 hours
    # and the final 12 hours near ICU discharge.
    los_hours = cohort_df['icu_los_hours'].fillna(0).clip(lower=0)
    total_windows = np.ceil(los_hours / 12).astype('int64')
    candidate_windows = (np.floor((los_hours - 12).clip(lower=0) / 12).astype('int64') - 4).clip(lower=0)
    return total_windows, candidate_windows

def _cohort_counts(step, cohort_df):
    total_windows, candidate_windows = _cohort_window_counts(cohort_df)
    return {
        'step': step,
        'n_subjects': int(cohort_df['subject_id'].nunique()),
        'n_hadm': int(cohort_df['hadm_id'].nunique()),
        'n_stays': int(cohort_df['stay_id'].nunique()),
        'total_12h_windows': int(total_windows.sum()),
        'candidate_12h_windows_excl_first48_last12': int(candidate_windows.sum()),
    }

adm_for_cohort = adm_pat_icu.copy()
for col in ['intime', 'outtime']:
    adm_for_cohort[col] = pd.to_datetime(adm_for_cohort[col], errors='coerce')
adm_for_cohort['icu_los_hours'] = (adm_for_cohort['outtime'] - adm_for_cohort['intime']).dt.total_seconds() / 3600

cohort_flow_rows = []
cohort = adm_for_cohort.copy()
cohort_flow_rows.append(_cohort_counts('0. All ICU stays from extraction', cohort))

cohort = cohort[cohort['icu_los_hours'] >= 72].copy()
cohort_flow_rows.append(_cohort_counts('1. ICU LOS >= 72 hours', cohort))

delirium_assessments_after_72h = (
    timeseries
    .loc[timeseries['feature_name'].eq('delirium') & timeseries['charttime'].notna(), ['stay_id', 'charttime']]
    .merge(adm_for_cohort[['stay_id', 'intime']], on='stay_id', how='left')
)
delirium_assessments_after_72h['hours_since_icu_admit'] = (
    delirium_assessments_after_72h['charttime'] - delirium_assessments_after_72h['intime']
).dt.total_seconds() / 3600
stays_with_delirium_assessment_after_72h = set(
    delirium_assessments_after_72h
    .loc[delirium_assessments_after_72h['hours_since_icu_admit'] > 72, 'stay_id']
)
cohort = cohort[cohort['stay_id'].isin(stays_with_delirium_assessment_after_72h)].copy()
cohort_flow_rows.append(_cohort_counts('2. >=1 delirium assessment after 72 hours', cohort))

cohort_final = cohort.copy()
cohort_final['ever_delirium'] = cohort_final['subject_id'].map(subject_ever_delirium).fillna(0).astype('int8')
cohort_stay_ids = set(cohort_final['stay_id'])
timeseries_cohort = timeseries[timeseries['stay_id'].isin(cohort_stay_ids)].copy()
wide_timeseries_cohort = wide_timeseries[wide_timeseries['stay_id'].isin(cohort_stay_ids)].copy()
wide_timeseries_12h_bin_cohort = wide_timeseries_12h_bin[wide_timeseries_12h_bin['stay_id'].isin(cohort_stay_ids)].copy()

cohort_flow = pd.DataFrame(cohort_flow_rows)
cohort_flow['stays_removed_from_previous'] = cohort_flow['n_stays'].shift(1).fillna(cohort_flow['n_stays']).astype(int) - cohort_flow['n_stays'].astype(int)
cohort_flow['stays_pct_of_initial'] = cohort_flow['n_stays'] / cohort_flow.loc[0, 'n_stays'] * 100

# Cohort-filtered outputs
timeseries_cohort.to_csv(OUTPUT_DIR / 'events_12h_long.csv', index=False)
wide_timeseries_cohort.to_csv(OUTPUT_DIR / 'events_12h_wide_by_charttime.csv', index=False)
wide_timeseries_12h_bin_cohort.to_csv(OUTPUT_DIR / 'events_12h_binned.csv', index=False)
cohort_final.to_csv(OUTPUT_DIR / 'cohort_final.csv', index=False)
cohort_flow.to_csv(OUTPUT_DIR / 'cohort_attrition.csv', index=False)

print(f"Cohort long-format events: {timeseries_cohort.shape}")
print(f"Saved: events_12h_long.csv ({len(timeseries_cohort):,} rows, {timeseries_cohort['stay_id'].nunique()} stays)")
print(f"Saved: events_12h_wide_by_charttime.csv ({len(wide_timeseries_cohort):,} rows, {wide_timeseries_cohort['stay_id'].nunique()} stays)")
print(f"Saved: events_12h_binned.csv ({len(wide_timeseries_12h_bin_cohort):,} rows, {wide_timeseries_12h_bin_cohort['stay_id'].nunique()} stays)")
print(f"Saved: cohort_final.csv ({len(cohort_final):,} stays)")
print(f"Saved: cohort_attrition.csv ({len(cohort_flow):,} rows)")
display(cohort_flow)

